In [ ]:
# Installo librerie necessarie

!pip install mplfinance
!pip install alpha_vantage


In [ ]:
from alpha_vantage.timeseries import TimeSeries
from alpha_vantage.foreignexchange import ForeignExchange
from alpha_vantage.techindicators import TechIndicators
from alpha_vantage.cryptocurrencies import CryptoCurrencies
from alpha_vantage.options import Options
from alpha_vantage.alphaintelligence import AlphaIntelligence
from alpha_vantage.fundamentaldata import FundamentalData
from alpha_vantage.commodities import Commodities
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
from datetime import datetime
from StochEq import StochasticEquations #First Download this file from the right repository
import datetime as dt
from statistics import correlation
import warnings
warnings.filterwarnings("ignore")
API = 'your-api'


In [ ]:
#Download data

# Initialize AlphaVantage Client
ts = TimeSeries(key=API, output_format='pandas')
fx = ForeignExchange(key=API, output_format='pandas')

# Download SPY ETFs historical data
ticker = 'SPY'
P, meta_data = ts.get_daily(symbol=ticker, outputsize='full')  # 'full' for total recent data
P.columns = ["Open","High","Low","Close","Volume"]
P = P.sort_index()

# Plot
mpf.plot(
    P[P.index >= datetime(2024,6,1)],
    type='candle',
    volume=True,
    style='charles',
    title=f'Grafico {ticker} YTD',
    figratio=(12,6),
    figscale=1.2
)

#Downloading Option Chain, choosing time interval and expiration time
op = Options(key=API,output_format='pandas')
chain, meta_data = op.get_historical_options(symbol="SPY")

exp_date = chain["expiration"].unique()
print(f"Available expiration date: \n{exp_date}")

In [ ]:
# Option
expiration_date = datetime(2025,4,9)
long_term = datetime(2026,3,20)
timeToMaturity = (expiration_date - datetime.today()).days
timeToLongTerm = (long_term - datetime.today()).days

call = chain[(chain["type"] == "call") & (chain["expiration"]==expiration_date.strftime('%Y-%m-%d'))]
callLT = chain[(chain["type"] == "call") & (chain["expiration"]==long_term.strftime('%Y-%m-%d'))]
put = chain[(chain["type"] == "put") & (chain["expiration"]==expiration_date.strftime('%Y-%m-%d'))]
putLT = chain[(chain["type"] == "put") & (chain["expiration"]==long_term.strftime('%Y-%m-%d'))]

columns_to_convert = ["strike","last","open_interest","implied_volatility","delta","gamma","theta","vega","rho"]

call[columns_to_convert] = call[columns_to_convert].apply(pd.to_numeric, errors="coerce")
callLT[columns_to_convert] = callLT[columns_to_convert].apply(pd.to_numeric, errors="coerce")
put[columns_to_convert] = put[columns_to_convert].apply(pd.to_numeric, errors="coerce")
putLT[columns_to_convert] = putLT[columns_to_convert].apply(pd.to_numeric, errors="coerce")

In [ ]:
##################################################################################
# Application of the SE framework to check the validity of Greeks
# We don't have to run this block cause AlphaVantage provide yet the Greeks values
##################################################################################

#Features
P["Return"] = P["Close"].pct_change().dropna()
P["Weekly Std Return"] = P["Return"].rolling(5).std()              #5 is a trading week
P["Monthly Std Return"] = P["Return"].rolling(22).std()            #22 is a trading month
P = P.dropna()

Ssigma = P["Return"].std()
trend = 0.000
S0 = P["Close"].iloc[-1]
closest_strike = call["strike"].iloc[(call["strike"] - S0).abs().argmin()]
closest_strikeLT = callLT["strike"].iloc[(callLT["strike"] - S0).abs().argmin()]
v0 = (call["implied_volatility"][call["strike"] == closest_strike]).values[0] #P["Monthly Std Return"].iloc[-1]
theta = (callLT["implied_volatility"][callLT["strike"] == closest_strikeLT]).values[0] #P["Weekly Std Return"].mean()
Vsigma = P["Monthly Std Return"].std()
rho = correlation(P["Close"],P["Monthly Std Return"])


# Parameters
T = timeToMaturity / 365                             # Time horizon (in year or fraction of years)
N = timeToMaturity                                   # Number of steps (daily)
M = 1000                                             # Number of simulations
t = 0.0 / 365                                        # Time at valuation (in year or fraction of years)
r = 0.0200 / 365                                     # Risk-free on a daily range (Hp: 3% for 1Y Generical Bond)
k = 1                                                # k = 1 for the speed of return to the long-term average

# Stock Paths with both Black-Scholes and Heston model
if timeToMaturity > 0:
    BSstock = StochasticEquations.BlackScholes(S0,r,Ssigma,0,T,N=timeToMaturity,M=5000,persistance=False)
    HSstock, StochVol = StochasticEquations.Heston(S0,v0,r,trend,k,theta,Vsigma,rho,T,N,M,return_vol=True)
    StochasticEquations.PlotHmodel(HSstock,S0,M)
    StochasticEquations.PlotBSmodel(BSstock,S0,M)
    plt.figure(figsize=(12,6))

    for i in range(M):
        plt.plot(StochVol[:,i],color="blue", alpha=0.1)

    plt.axhline(v0,linestyle = "--",label=f"Volatilità iniziale: {v0:.2f}",color="darkgrey")
    plt.title("Heston Model - Volatility Paths")
    plt.xlabel('Time Steps')
    plt.ylabel('Stochastic Volatility')
    plt.legend()
    plt.show()

else:
    pass



#Plot the delta graph
plt.rcParams["figure.figsize"] = (15,10)
ax1 = plt.subplot(2,2,1)
ax1.plot(call["strike"],call["delta"], linewidth = 1.5, color="darkgreen", label="Call delta")
ax1.plot(put["strike"],put["delta"], linewidth = 1.5, color="Blue", label="Put delta")
ax1.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax1.axhline(0,linestyle="dotted")
ax1.legend()
ax1.set_title(f"{ticker} delta")

#Plot the gamma graph
ax2 = plt.subplot(2,2,2)
ax2.plot(call["strike"],call["gamma"], linewidth = 1.5, color="darkgreen", label="Call gamma")
ax2.plot(put["strike"],put["gamma"], linewidth = 1.5, color="Blue", label="Put gamma")
ax2.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax2.axhline(0,linestyle="dotted")
ax2.legend()
ax2.set_title(f"{ticker} gamma")

#Plot the vega graph
ax3 = plt.subplot(2,2,3)
ax3.plot(call["strike"],call["vega"], linewidth = 1.5, color="darkgreen", label="Call vega")
ax3.plot(put["strike"],put["vega"], linewidth = 1.5, color="Blue", label="Put vega")
ax3.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax3.axhline(0,linestyle="dotted")
ax3.legend()
ax3.set_title(f"{ticker} vega")


#Plot the theta graph
ax4 = plt.subplot(2,2,4)
ax4.plot(call["strike"],call["theta"], linewidth = 1.5, color="darkgreen", label="Call theta")
ax4.plot(put["strike"],put["theta"], linewidth = 1.5, color="Blue", label="Put theta")
ax4.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax4.axhline(0,linestyle="dotted")
ax4.legend()
ax4.set_title(f"{ticker} theta")
plt.show()

#Plot Iv Call
plt.rcParams["figure.figsize"] = (15,10)
ax5 = plt.subplot(2,2,1)
ax5.plot(call["strike"],call["implied_volatility"], linewidth = 1.5, color="blue")
ax5.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax5.axhline(0,linestyle="dotted")
ax5.legend()
ax5.set_title("Call Implied Volatility")

#Plot Iv Put
ax6 = plt.subplot(2,2,2)
ax6.plot(put["strike"],put["implied_volatility"], linewidth = 1.5, color="blue")
ax6.axvline(S0,color="darkgrey",linestyle="--", label="Stock Price")
ax6.axhline(0,linestyle="dotted")
ax6.legend()
ax6.set_title("Put Implied Volatility")
plt.show()

In [ ]:
# Calculation of GEX and IV*GEX Indicator
# The Gamma Exposure is calculated multiplying Gamma x P Underlying x Open Interest x Stock for each contract
# In the same set of call/put contract the P Underlying and Stock for each contract are the same so we could
# Simply calculate the product between Gamma & Open Interest

call["GEX"] = call["gamma"] * call["open_interest"] * 100 * (S0 ** 2)
call["IV*GEX"] = call["implied_volatility"] * call["GEX"]
put["GEX"] = -1 * put["gamma"] * put["open_interest"] * 100 * (S0 ** 2)
put["IV*GEX"] = put["implied_volatility"] * put["GEX"]

# Calculation GEX by Strike for potential critic isitutional levels
df_all = pd.concat([call,put],ignore_index=True)

gex_by_strike = df_all.groupby("strike")["GEX"].sum().sort_values(ascending=False)

# Visualizza i top 5 livelli più "sticky"
print("Top 5 leves of positive GEX:")
print(gex_by_strike.head(5))

# Visualizza i top 5 livelli negativi (potenziali zone esplosive)
print("\nTop 5 leves of negative GEX:")
print(gex_by_strike.copy().sort_values().head(5))

print(f'\nsticky level for call option: {call.loc[call["IV*GEX"].idxmax(),"strike"]}')
print(f'sticky level for put option: {put.loc[np.abs(put["IV*GEX"]).idxmax(),"strike"]}')
print(f'Second sticky level for call option: {call.loc[call["IV*GEX"].nlargest(2).index[1],"strike"]}')
print(f'Second sticky level for put option: {put.loc[np.abs(put["IV*GEX"]).nlargest(2).index[1],"strike"]}')


SupRes=[call.loc[call["IV*GEX"].idxmax(),"strike"],
        put.loc[np.abs(put["IV*GEX"]).idxmax(),"strike"],
        call.loc[call["IV*GEX"].nlargest(2).index[1],"strike"],
        put.loc[np.abs(put["IV*GEX"]).nlargest(2).index[1],"strike"]]


fig, ax =mpf.plot(
    P[P.index >= datetime(2024,6,1)],
    hlines=dict(hlines=[SupRes[0],SupRes[1]], colors=['r','g'], linestyle=['--','--']),
    type='candle',
    volume=True,
    style='yahoo',
    title=f'Grafico {ticker} with Support and Resistance',
    figratio=(12,6),
    figscale=1.2,
    returnfig=True,
)

ANALYSIS OF GAMMA EXPOSURE BASED ON THE CURRENT TRADING WEEK AIMED TO FIND THE CRITIC LEVELS ON WHICH THE MARKET MAKERS HAVE MOST EXPOSURE SO THEY HAVE TO HEDGE THEMSELVES

In [ ]:
for date in exp_date[2:6]:
    print("/*----------------------------------------*/")
    print(f"\nLoop for GEX levels at expiration: {date}")

    callGEX = chain[(chain["type"] == "call") & (chain["expiration"]==date)]
    putGEX = chain[(chain["type"] == "put") & (chain["expiration"]==date)]

    callGEX[columns_to_convert] = callGEX[columns_to_convert].apply(pd.to_numeric, errors="coerce")
    putGEX[columns_to_convert] = putGEX[columns_to_convert].apply(pd.to_numeric, errors="coerce")

    callGEX["GEX"] = callGEX["gamma"] * callGEX["open_interest"] * 100 * (S0 ** 2)
    callGEX["IV*GEX"] = callGEX["implied_volatility"] * callGEX["GEX"]
    putGEX["GEX"] = -1 * putGEX["gamma"] * putGEX["open_interest"] * 100 * (S0 ** 2)
    putGEX["IV*GEX"] = putGEX["implied_volatility"] * putGEX["GEX"]

    # Calculation GEX by Strike for potential critic isitutional levels
    df_all = pd.concat([callGEX,putGEX],ignore_index=True)

    gex_by_strike = df_all.groupby("strike")["GEX"].sum().sort_values(ascending=False)

    # Visualizza i top 5 livelli più "sticky"
    print(f"\nTop 5 leves of positive GEX at {date}:")
    print(f"{gex_by_strike.head(5)}")

    # Visualizza i top 5 livelli negativi (potenziali zone esplosive)
    print(f"\nTop 5 leves of negative GEX at {date}:")
    print(f"{gex_by_strike.copy().sort_values().head(5)}")

    print("/*----------------------------------------*/")

In [ ]:
# Short Strangle strategy Under Heston Model Hypotesis

short_call = call["strike"][call["strike"] == SupRes[0]].values[0]
short_put = put["strike"][put["strike"] == SupRes[1]].values[0]
premium_call = call["last"][call["strike"] == SupRes[0]].values[0]
premium_put = put["last"][put["strike"] == SupRes[1]].values[0]

if timeToMaturity <= 0:
    spot_prices = np.sort(call["strike"])
else:
    spot_prices = np.sort(HSstock[-1])

# Short call payoff
def payoff_call_sold(spot, strike, premium):
    return np.minimum(0, strike - spot) + premium

# Short put payoff
def payoff_put_sold(spot, strike, premium):
    return np.minimum(0, spot - strike) + premium

# Short Strangle Payoff
payoff_call = payoff_call_sold(spot_prices, short_call, premium_call) * 10
payoff_put = payoff_put_sold(spot_prices, short_put, premium_put) * 10
payoff_short_strangle = (payoff_call + payoff_put)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(spot_prices, payoff_call, label='Short Payoff Call (Strike {})'.format(short_call), linestyle='--')
plt.plot(spot_prices, payoff_put, label='Short Payoff Put (Strike {})'.format(short_put), linestyle='--')
plt.plot(spot_prices, payoff_short_strangle, label='Payoff Short Strangle', color='black', linewidth=2)
plt.title('Payoff Short Strangle')
plt.xlabel('Underlying price at expiry')
plt.ylabel('Payoff')
plt.axhline(0, color='black',linewidth=0.5)
plt.axvline(short_call, color='blue', linestyle='--', linewidth=0.7)
plt.axvline(short_put, color='blue', linestyle='--', linewidth=0.7)
plt.legend()
plt.grid(True)
plt.show()
